# GAN sencilla con PyTorch

El generador transforma ruido en imágenes y el discriminador aprende a distinguir imágenes reales de las generadas. Entrenamos con una muestra de MNIST para que el ejercicio sea breve.

## Entorno IPD434

Usa el kernel **IPD434**. MNIST se descarga una vez en `../data/mnist`. CUDA se usa si está disponible.

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| dispositivo:', device)

## Datos y ruido

Las imágenes se escalan a $[0,1]$, coherente con la salida sigmoide del generador. Limitamos el número de ejemplos para reducir el tiempo de entrenamiento.

In [ ]:
DATA_ROOT = Path('../data/mnist')
dataset = datasets.MNIST(DATA_ROOT, train=True, download=True, transform=transforms.ToTensor())
subset_indices = torch.randperm(len(dataset), generator=torch.Generator().manual_seed(SEED))[:12000].tolist()
train_set = Subset(dataset, subset_indices)
train_loader = DataLoader(
    train_set, batch_size=128, shuffle=True,
    generator=torch.Generator().manual_seed(SEED), num_workers=0,
    pin_memory=device.type == 'cuda'
)
print('Imágenes de entrenamiento:', len(train_set))

## Generador y discriminador

El generador amplía un vector latente de 100 valores hasta una imagen de $28\times28$. El discriminador produce un logit, cuyo signo ayuda a decidir si la imagen parece real.

In [ ]:
LATENT_DIM = 100

class Generator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.project = nn.Sequential(
            nn.Linear(latent_dim, 7 * 7 * 128),
            nn.BatchNorm1d(7 * 7 * 128),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.upsample = nn.Sequential(
            nn.Unflatten(1, (128, 7, 7)),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(64, 1, 4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, z):
        return self.upsample(self.project(z))

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 64, 5, stride=2, padding=2),
            nn.LeakyReLU(0.2, inplace=True), nn.Dropout(0.3),
            nn.Conv2d(64, 128, 5, stride=2, padding=2),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3), nn.Flatten(),
            nn.Linear(128 * 7 * 7, 1),
        )

    def forward(self, images):
        return self.features(images).squeeze(1)

generator = Generator().to(device)
discriminator = Discriminator().to(device)
criterion = nn.BCEWithLogitsLoss()
g_optimizer = torch.optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
d_optimizer = torch.optim.Adam(discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))
print('Generador:', tuple(generator(torch.randn(4, LATENT_DIM, device=device)).shape))

## Antes de entrenar

Con pesos aleatorios, el generador produce patrones sin sentido. La señal de entrenamiento adversarial guía sus parámetros para parecerse a las imágenes de MNIST.

In [ ]:
def show_generated(model, count=16, title='Imágenes generadas'):
    model.eval()
    with torch.no_grad():
        noise = torch.randn(count, LATENT_DIM, device=device)
        images = model(noise).cpu()
    fig, axes = plt.subplots(4, 4, figsize=(5, 5))
    for ax, image in zip(axes.ravel(), images):
        ax.imshow(image[0], cmap='gray')
        ax.axis('off')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

show_generated(generator, title='Generador antes de entrenar')

## Entrenamiento adversarial

Primero actualizamos el discriminador con imágenes reales y falsas. Luego actualizamos el generador para que el discriminador clasifique como reales las imágenes generadas.

In [ ]:
EPOCHS = 5
g_losses, d_losses = [], []
for epoch in range(1, EPOCHS + 1):
    generator.train()
    discriminator.train()
    g_sum, d_sum, n_batches = 0.0, 0.0, 0
    for real_images, _ in train_loader:
        real_images = real_images.to(device, non_blocking=True)
        batch_size = real_images.size(0)
        real_targets = torch.ones(batch_size, device=device)
        fake_targets = torch.zeros(batch_size, device=device)

        # Actualizar discriminador
        noise = torch.randn(batch_size, LATENT_DIM, device=device)
        fake_images = generator(noise)
        d_optimizer.zero_grad(set_to_none=True)
        real_logits = discriminator(real_images)
        fake_logits = discriminator(fake_images.detach())
        d_loss = criterion(real_logits, real_targets) + criterion(fake_logits, fake_targets)
        d_loss.backward()
        d_optimizer.step()

        # Actualizar generador para que las imágenes parezcan reales
        noise = torch.randn(batch_size, LATENT_DIM, device=device)
        g_optimizer.zero_grad(set_to_none=True)
        generated_logits = discriminator(generator(noise))
        g_loss = criterion(generated_logits, real_targets)
        g_loss.backward()
        g_optimizer.step()

        d_sum += d_loss.item()
        g_sum += g_loss.item()
        n_batches += 1
    d_losses.append(d_sum / n_batches)
    g_losses.append(g_sum / n_batches)
    print(f'Época {epoch}/{EPOCHS} | D={d_losses[-1]:.3f} | G={g_losses[-1]:.3f}')
    show_generated(generator, title=f'Época {epoch}')

In [ ]:
plt.plot(d_losses, label='discriminador')
plt.plot(g_losses, label='generador')
plt.xlabel('Época')
plt.ylabel('BCE')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Para interpretar

Las pérdidas del generador y el discriminador compiten, por lo que no necesariamente disminuyen juntas. Evalúa también la variedad y calidad de las imágenes generadas.